# Week 10 — Run a small interaction repeatedly

**Research task:** Compare a no-interaction baseline with a condition in which each simulated actor sees the preceding action.

**Python introduced:** repeated function calls, `range(...)`, accumulated run lists, conditions and small outcome summaries.

This is the runnable coding component. The full literature-led chapter and slides remain to be developed.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/christopherbarrie/GenAI_Soc2026/blob/main/workbook/session10/session10_collective_intelligence.ipynb)

Colab supports OpenRouter only; use local Jupyter for dual-route work.

In [ ]:
# Colab setup for the OpenRouter route.
import os as setup_os
import subprocess as setup_subprocess
from pathlib import Path as SetupPath
if SetupPath('/content').exists():
    setup_repo=SetupPath('/content/GenAI_Soc2026')
    if not setup_repo.exists(): setup_subprocess.run(['git','clone','https://github.com/cjbarrie/GenAI_Soc2026.git',str(setup_repo)],check=True)
    setup_os.chdir(setup_repo/'workbook'/'session10')
print('Working folder:',SetupPath.cwd())

## Load the course settings and SDKs

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
import json
import os
from getpass import getpass
from pathlib import Path

import ollama
from openrouter import OpenRouter

ROOT = Path.cwd()
while not (ROOT / "config" / "course_models.json").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
config = json.loads((ROOT / "config" / "course_models.json").read_text())
HOSTED_MODEL = config["hosted"]["model"]
LOCAL_MODEL = config["local"]["model"]
if not os.getenv("OPENROUTER_API_KEY"):
    os.environ["OPENROUTER_API_KEY"] = getpass("OpenRouter course key (hidden): ")

## Choose a route and define the action schema

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
ROUTE = "ollama"
action_schema = {
    "type":"object","properties":{"action":{"type":"string"}},
    "required":["action"],"additionalProperties":False,
}

## Define one actor update function

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
def choose_action(actor, observation, route):
    messages = [{"role":"user","content":(
        "Choose JOIN or STAY_OUT for this simulated actor. Actor: " + json.dumps(actor) +
        " Observation: " + observation + " Return JSON."
    )}]
    if route == "openrouter":
        with OpenRouter(api_key=os.environ["OPENROUTER_API_KEY"]) as client:
            response = client.chat.send(
                model=HOSTED_MODEL,messages=messages,temperature=0,
                response_format={"type":"json_schema","json_schema":{
                    "name":"join_action","strict":True,"schema":action_schema,
                }},
            )
        raw_output = response.choices[0].message.content
    else:
        response = ollama.chat(model=LOCAL_MODEL,messages=messages,format=action_schema,options={"temperature":0})
        raw_output = response.message.content
    return json.loads(raw_output)["action"]

## Store three actors and two conditions

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
actors = [
    {"id":"a","initial_support":"low"},
    {"id":"b","initial_support":"medium"},
    {"id":"c","initial_support":"high"},
]
conditions = ["baseline", "interaction"]
all_runs = []

## Repeat each condition three times

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for condition in conditions:
    for run_number in range(3):
        actions = []
        previous_action = "No prior action is visible."
        for actor in actors:
            if condition == "baseline":
                observation = "No other actor's action is visible."
            else:
                observation = "The preceding actor chose: " + previous_action
            action = choose_action(actor, observation, ROUTE)
            actions.append(action)
            previous_action = action
        all_runs.append({"condition":condition,"run":run_number,"actions":actions})
        print(condition, run_number, actions)

## Compare convergence without selecting one favorite run

Before running, name the input and predict the output. After running, explain what the block added to the research record.

In [ ]:
for record in all_runs:
    all_same = len(set(record["actions"])) == 1
    print(record["condition"], record["run"], "all same:", all_same)

# ONE CHANGE: reverse the actor order and rerun all conditions.

## Methodological check

Repeated convergence is not automatically collective intelligence. Compare the baseline, preserve order and consider shared model priors, prompt effects and leakage.
## Recording

Explain one actor update, one full group run and how `range(3)` creates repeated evidence. Reverse actor order and discuss what the changed or unchanged outcomes establish.